In [ ]:
# IMPORTS AND SETTINGS
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


INPUT_FILE = "Parts Training Usage.csv"
OUTPUT_FOLDER = Path("forecast_results")

# Fixed historical backtest:
# Train through December 2025 and predict January-June 2026.
TRAINING_CUTOFF = "2026-01-01"
FORECAST_MONTHS = 6

TARGET_COLUMN = "Monthly Inventory Issues"
PART_COLUMN = "fpartno"
DATE_COLUMN = "Date"

LAGS = [1, 2, 3, 6, 12]

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

print("Settings loaded.")


In [ ]:
# LOAD AND PREPARE DATA
data = pd.read_csv(INPUT_FILE)

required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

data[DATE_COLUMN] = pd.to_datetime(
    data[DATE_COLUMN],
    errors="raise",
)

data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN],
    errors="raise",
)

# Keep January 2023 and later.
data = data[
    data[DATE_COLUMN] >= pd.Timestamp("2023-01-01")
].copy()

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)

if duplicates.any():
    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows were found:\n"
        f"{duplicate_rows.head(20)}"
    )

print(f"Rows loaded: {len(data):,}")
print(
    f"Unique parts: "
    f"{data[PART_COLUMN].nunique():,}"
)
print(
    f"First date: "
    f"{data[DATE_COLUMN].min()}"
)
print(
    f"Last date: "
    f"{data[DATE_COLUMN].max()}"
)


In [ ]:
# CREATE THE FEATURES
def create_training_features(
    historical_data,
):

    feature_data = historical_data.copy()

    # Date and time features
    feature_data["month"] = (
        feature_data[DATE_COLUMN].dt.month
    )

    feature_data["year"] = (
        feature_data[DATE_COLUMN].dt.year
    )

    feature_data["quarter"] = (
        feature_data[DATE_COLUMN].dt.quarter
    )

    feature_data["time_idx"] = np.arange(
        len(feature_data)
    )

    # Lag features
    for lag in LAGS:
        feature_data[f"lag_{lag}"] = (
            feature_data[TARGET_COLUMN].shift(lag)
        )

    # Shift usage so the current month's actual usage
    # is never used to predict itself.
    prior_usage = (
        feature_data[TARGET_COLUMN].shift(1)
    )

    # Rolling averages
    feature_data["rolling_mean_3"] = (
        prior_usage.rolling(3).mean()
    )

    feature_data["rolling_mean_6"] = (
        prior_usage.rolling(6).mean()
    )

    feature_data["rolling_mean_12"] = (
        prior_usage.rolling(12).mean()
    )

    # Rolling medians
    feature_data["rolling_median_3"] = (
        prior_usage.rolling(3).median()
    )

    feature_data["rolling_median_6"] = (
        prior_usage.rolling(6).median()
    )

    feature_data["rolling_median_12"] = (
        prior_usage.rolling(12).median()
    )

    # Total usage over the prior 12 months
    feature_data["rolling_total_12"] = (
        prior_usage.rolling(12).sum()
    )

    # Demand variability
    feature_data["rolling_std_3"] = (
        prior_usage.rolling(3).std()
    )

    feature_data["rolling_std_6"] = (
        prior_usage.rolling(6).std()
    )

    feature_data["rolling_std_12"] = (
        prior_usage.rolling(12).std()
    )

    # Annual demand range
    feature_data["rolling_min_12"] = (
        prior_usage.rolling(12).min()
    )

    feature_data["rolling_max_12"] = (
        prior_usage.rolling(12).max()
    )

    # Percentage of prior 12 months
    # with zero usage
    feature_data[
        "zero_month_percentage_12"
    ] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values: (
                values == 0
            ).mean(),
            raw=True,
        )
    )

    # Demand variability relative to
    # average demand
    feature_data[
        "coefficient_variation_12"
    ] = (
        feature_data["rolling_std_12"]
        /
        feature_data[
            "rolling_mean_12"
        ].replace(0, np.nan)
    )

    # Recent usage compared with
    # the annual monthly average
    feature_data["recent_vs_annual"] = (
        feature_data["rolling_mean_3"]
        -
        feature_data["rolling_mean_12"]
    )

    # Demand trends
    feature_data["trend_3"] = (
        feature_data["lag_1"]
        -
        feature_data["lag_3"]
    )

    feature_data["trend_6"] = (
        feature_data["lag_1"]
        -
        feature_data["lag_6"]
    )

    feature_columns = [
        "month",
        "year",
        "quarter",
        "time_idx",

        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",

        "rolling_mean_3",
        "rolling_mean_6",
        "rolling_mean_12",

        "rolling_median_3",
        "rolling_median_6",
        "rolling_median_12",

        "rolling_total_12",

        "rolling_std_3",
        "rolling_std_6",
        "rolling_std_12",

        "rolling_min_12",
        "rolling_max_12",

        "zero_month_percentage_12",
        "coefficient_variation_12",
        "recent_vs_annual",

        "trend_3",
        "trend_6",
    ]

    training_rows = (
        feature_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=feature_columns
        )
        .reset_index(drop=True)
    )

    return (
        training_rows,
        feature_columns,
    )


print(
    "Version 2 feature function created."
)


In [ ]:
# TRAIN THE LIGHTGBM MODEL
def train_model(
    training_rows,
    feature_columns,
):

    model = LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        training_rows[feature_columns],
        training_rows[TARGET_COLUMN],
    )

    return model


print("Training function created.")


In [ ]:
# GENERATIVE THE RECURSIVE FORECAST
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
            ]
        ]
        .copy()
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    predictions = []

    for _ in range(forecast_months):

        next_date = (
            forecast_history[
                DATE_COLUMN
            ].max()
            + pd.DateOffset(months=1)
        )

        recent_usage = (
            forecast_history[TARGET_COLUMN]
        )

        future_row = {
            "month": next_date.month,
            "year": next_date.year,
            "quarter": next_date.quarter,
            "time_idx": len(
                forecast_history
            ),
        }

        # Lag features
        for lag in LAGS:
            future_row[f"lag_{lag}"] = (
                recent_usage.iloc[-lag]
            )

        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]

        # Rolling averages
        future_row["rolling_mean_3"] = (
            last_3.mean()
        )

        future_row["rolling_mean_6"] = (
            last_6.mean()
        )

        future_row["rolling_mean_12"] = (
            last_12.mean()
        )

        # Rolling medians
        future_row["rolling_median_3"] = (
            last_3.median()
        )

        future_row["rolling_median_6"] = (
            last_6.median()
        )

        future_row["rolling_median_12"] = (
            last_12.median()
        )

        # Annual usage
        future_row["rolling_total_12"] = (
            last_12.sum()
        )

        # Standard deviations
        future_row["rolling_std_3"] = (
            last_3.std()
        )

        future_row["rolling_std_6"] = (
            last_6.std()
        )

        future_row["rolling_std_12"] = (
            last_12.std()
        )

        # Annual range
        future_row["rolling_min_12"] = (
            last_12.min()
        )

        future_row["rolling_max_12"] = (
            last_12.max()
        )

        # Zero-usage percentage
        future_row[
            "zero_month_percentage_12"
        ] = last_12.eq(0).mean()

        rolling_mean_12 = (
            future_row["rolling_mean_12"]
        )

        rolling_std_12 = (
            future_row["rolling_std_12"]
        )

        # Coefficient of variation
        if rolling_mean_12 != 0:
            future_row[
                "coefficient_variation_12"
            ] = (
                rolling_std_12
                / rolling_mean_12
            )
        else:
            future_row[
                "coefficient_variation_12"
            ] = 0.0

        # Recent versus annual demand
        future_row["recent_vs_annual"] = (
            future_row["rolling_mean_3"]
            -
            future_row["rolling_mean_12"]
        )

        # Trend features
        future_row["trend_3"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-3]
        )

        future_row["trend_6"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-6]
        )

        future_features = pd.DataFrame(
            [future_row],
            columns=feature_columns,
        )

        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )

        # Usage cannot be negative.
        predicted_usage = max(
            0.0,
            predicted_usage,
        )

        predictions.append(
            {
                PART_COLUMN: part_number,
                DATE_COLUMN: next_date,
                "Predicted Usage": (
                    predicted_usage
                ),
            }
        )

        # Use the decimal prediction internally
        # for the next forecast month.
        new_history_row = pd.DataFrame(
            {
                DATE_COLUMN: [next_date],
                TARGET_COLUMN: [
                    predicted_usage
                ],
            }
        )

        forecast_history = pd.concat(
            [
                forecast_history,
                new_history_row,
            ],
            ignore_index=True,
        )

    return pd.DataFrame(predictions)


print(
    "Version 2 forecast function created."
)


In [ ]:
# TRAIN THROUGH 2025 AND PREDICT JAN-JUNE 2026
cutoff_date = pd.Timestamp(
    TRAINING_CUTOFF
)

forecast_end = (
    cutoff_date
    + pd.DateOffset(
        months=FORECAST_MONTHS
    )
)

all_results = []
errors = []

for part_number in sorted(
    data[PART_COLUMN]
    .dropna()
    .unique()
):

    print(
        f"\nProcessing: {part_number}"
    )

    try:
        part_data = (
            data[
                data[PART_COLUMN]
                == part_number
            ]
            .copy()
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )

        # Training data ends with
        # December 2025.
        historical_data = (
            part_data[
                part_data[DATE_COLUMN]
                < cutoff_date
            ]
            .copy()
        )

        # Actual January-June 2026 values
        actual_data = (
            part_data[
                (
                    part_data[DATE_COLUMN]
                    >= cutoff_date
                )
                &
                (
                    part_data[DATE_COLUMN]
                    < forecast_end
                )
            ][
                [
                    DATE_COLUMN,
                    TARGET_COLUMN,
                ]
            ]
            .copy()
        )

        if len(actual_data) != FORECAST_MONTHS:
            raise ValueError(
                f"Expected "
                f"{FORECAST_MONTHS} "
                f"actual validation months, "
                f"but found "
                f"{len(actual_data)}."
            )

        training_rows, feature_columns = (
            create_training_features(
                historical_data
            )
        )

        if training_rows.empty:
            raise ValueError(
                "No usable training rows "
                "after feature creation."
            )

        model = train_model(
            training_rows,
            feature_columns,
        )

        forecast = forecast_future_months(
            model=model,
            historical_data=historical_data,
            feature_columns=feature_columns,
            forecast_months=FORECAST_MONTHS,
            part_number=part_number,
        )

        comparison = forecast.merge(
            actual_data,
            on=DATE_COLUMN,
            how="left",
        )

        comparison = comparison.rename(
            columns={
                TARGET_COLUMN: (
                    "Actual Usage"
                )
            }
        )

        comparison["Error"] = (
            comparison["Predicted Usage"]
            -
            comparison["Actual Usage"]
        )

        comparison["Absolute Error"] = (
            comparison["Error"].abs()
        )

        comparison["Training Through"] = (
            historical_data[
                DATE_COLUMN
            ].max()
        )

        all_results.append(comparison)

        print(
            f"Finished: {part_number}"
        )

    except Exception as error:

        errors.append(
            {
                PART_COLUMN: (
                    str(part_number)
                ),
                "Error": str(error),
            }
        )

        print(
            f"Skipped {part_number}: "
            f"{error}"
        )


if not all_results:
    raise RuntimeError(
        "No Version 2 forecasts "
        "completed successfully."
    )

print(
    "\nVersion 2 backtest complete."
)


In [ ]:
# DISPLAY PREDICTIONS WITH ACTUAL USAGE
results = (
    pd.concat(
        all_results,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

results["Predicted Usage Rounded"] = (
    results["Predicted Usage"]
    .round()
    .astype(int)
)

results["Actual Usage Rounded"] = (
    results["Actual Usage"]
    .round()
    .astype(int)
)

display(
    results[
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Training Through",
            "Predicted Usage Rounded",
            "Actual Usage Rounded",
            "Predicted Usage",
            "Actual Usage",
            "Error",
            "Absolute Error",
        ]
    ]
)


In [ ]:
# OPTIONAL: CALCULATES OVERALL MAE
average_absolute_error = (
    results["Absolute Error"].mean()
)

print(
    "Version 2 Average Absolute Error: "
    f"{average_absolute_error:.2f} units"
)


In [ ]:
# ACCURACY BY PART
part_accuracy = (
    results
    .groupby(PART_COLUMN)
    .agg(
        Average_Actual_Usage=(
            "Actual Usage",
            "mean",
        ),
        Average_Predicted_Usage=(
            "Predicted Usage",
            "mean",
        ),
        Actual_Total=(
            "Actual Usage",
            "sum",
        ),
        Predicted_Total=(
            "Predicted Usage",
            "sum",
        ),
        MAE=(
            "Absolute Error",
            "mean",
        ),
    )
    .reset_index()
)

part_accuracy[
    "Six_Month_Total_Error"
] = (
    part_accuracy["Predicted_Total"]
    -
    part_accuracy["Actual_Total"]
)

display(part_accuracy)
